### **Installs and Imports**

In [ ]:
!pip install -q transformers datasets peft

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [ ]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",                # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                       # dropout on the adapter path
    bias           = "none",                     # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **CPT-LoRA** using HuggingFace wikitext-dataset:

In [ ]:
from datasets import load_dataset
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# CPT = raw text, every token counts. Join non-empty lines into one corpus.
train_text = "\n".join(t for t in ds["train"]["text"]      if t.strip())
val_text   = "\n".join(t for t in ds["validation"]["text"] if t.strip())

# Pack into one flat stream — exactly your Shakespeare CPT prep, new source
train_ids = torch.tensor(tokenizer(train_text, add_special_tokens=False)["input_ids"])
val_ids   = torch.tensor(tokenizer(val_text,   add_special_tokens=False)["input_ids"])
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train tokens: 2,543,191 | val tokens: 265,683


### **Hyperparameters + The Batch Loader:**

In [ ]:
block_size, batch_size = 256, 8
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 400, 50

def get_batch(split):
    d  = train_ids if split == "train" else val_ids
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i:i+block_size] for i in ix])
    return xb.to(device)

### **Optimizer** + fixed held-out eval:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb = get_batch("val")
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

### **Training loop:**

In [ ]:
model.train()
for step in range(max_steps):
    xb   = get_batch("train")
    loss = model(input_ids=xb, labels=xb).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 3.1201 | val 3.3997
step   50 | train 3.1901 | val 3.2622
step  100 | train 2.9237 | val 3.2301
step  150 | train 3.3488 | val 3.2001
step  200 | train 3.3170 | val 3.1246
step  250 | train 2.9052 | val 3.1105
step  300 | train 3.4282 | val 3.0979
step  350 | train 3.0690 | val 3.1290
step  399 | train 2.7710 | val 3.0614


### **Generate:**

In [ ]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("The history of the Roman Empire"))

### Before **LoRA-CPT**:

The history of the Roman Empire began in 286 BC, when a large group of people from what is now Turkey were expelled by the Romans. They settled around the city of Rome and became known as the "Romans". The Romans had many different cultures that influenced their way of life - Greek culture was very important to them because it helped shape how they saw themselves today!
One interesting thing about the Romans' relationship with other ancient civilizations like Greece comes up again: there are some similarities between these two groups but also lots more differences too (like language). For example; while Greeks spoke Latin instead of Ancient Greek due its

### After **LoRA-CPT**:

The history of the Roman Empire has been a subject that has attracted much debate and discussion among scholars. There are many different theories as to how it all began, but historians generally agree on one thing: Rome was founded in 753 BC by Romulus , who ruled over what is now modern day Italy . After a series of wars between his brothers , he built a city called Rome which would eventually become the capital for an empire spanning three continents ( Eurasia + Africa ) from its founding around 201 BCE until it fell under Muslim rule during the sixth century CE . The Romans were known throughout Europe due their advancements across

### **Save the Model:**

In [ ]:
model.save_pretrained("smollm-lora-cpt-wikitext")   # saves ONLY the adapters — a few MB, not 500MB

### **Reload and add the adapters:**

In [ ]:
from peft import PeftModel
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-cpt-wikitext").to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]